# Giai đoạn 4 — Bước 5A: Evidence & Provenance Gate

**Notebook để bạn tự chạy.** Chưa gọi LLM, chưa sinh câu trả lời y khoa, không sửa ChromaDB, không huấn luyện lại ML. Mục đích: phân tích 42 đánh giá từ Bước 4, kiểm tra nguồn/trích dẫn, điều tra Q01/Q05, chuẩn bị điều kiện cho Bước 5B.

**Chú ý:** Các câu đã dùng trong Bước 4 là **development set**; nếu thay Retrieval dựa trên chúng, phải đánh giá lại trên các câu mới để tránh báo cáo kết quả lạc quan.

## 0. Chuẩn bị các file và kiểm tra trạng thái
Không chạy lại notebook xuất CSV 42 nhãn nếu đã chấm.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, json, hashlib, math

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd,*cwd.parents) if (p/'registry'/'source_registry.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook trong cây thư mục Giai đoạn 4 có registry/source_registry.csv')
STEP3 = ROOT/'data'/'outputs'/'embedding_chroma_step3'/'step3_embedding_chroma_report.json'
STEP4 = ROOT/'data'/'outputs'/'retrieval_eval_step4'/'step4_retrieval_eval_report.json'
REVIEW4 = ROOT/'data'/'outputs'/'retrieval_eval_step4'/'step4_retrieval_review.csv'
OUT = ROOT/'data'/'outputs'/'rag_gate_step5a'
OUT.mkdir(parents=True,exist_ok=True)
RIGHTS_TEMPLATE = ROOT/'registry'/'step5a_source_rights_review.csv'
SOURCE_AUDIT = OUT/'step5a_source_audit.csv'
TOP10_CSV = OUT/'step5a_q01_q05_top10_review.csv'
GATE_JSON = OUT/'step5a_gate_report.json'
for p in [STEP3,STEP4,REVIEW4]:
    assert p.is_file(), f'Thiếu file trên máy bạn: {p}'
r3=json.loads(STEP3.read_text(encoding='utf-8'))
r4=json.loads(STEP4.read_text(encoding='utf-8'))
with REVIEW4.open(encoding='utf-8-sig',newline='') as f:
    rows=list(csv.DictReader(f))
assert len(rows)==42, f'Cần 42 dòng chấm, có {len(rows)}'
assert len({(r['query_id'],r['rank']) for r in rows})==42
assert r3['collection']==r4['collection'] and r3['records_indexed']==r4['indexed_records_unchanged']==66
assert r4['status']=='human_labels_complete_research_only'
assert r4['queries_total']==14 and r4['queries_in_scope']==12
assert r4.get('step3_source_snapshot_sha256')==r3['source_snapshot_sha256']
print('DATA OK | Collection:',r3['collection'],'| Labeled:',len(rows),'| Sources:',len(r3['sources_indexed']))


DATA OK | Collection: diabetes_kb_step3_e5small_v1 | Labeled: 42 | Sources: 7


## 1. Audit trích dẫn và trạng thái bằng chứng theo từng nguồn

Đây là thống kê trên 42 **lượt chấm**; không được suy diễn một dấu `yes` trên hit thành giấy phép áp dụng cho mọi tài liệu.

In [2]:
grouped=defaultdict(list)
for row in rows:
    assert row['relevance_grade'] in ('0','1','2')
    assert row['citation_ok'].lower() in ('yes','no')
    assert row['rights_evidence_ok'].lower() in ('yes','no','unknown')
    grouped[row['source_id']].append(row)
audit=[]
for sid,nrec in sorted(r3['sources_indexed'].items()):
    rr=grouped.get(sid,[])
    x={'source_id':sid,'indexed_chunks':nrec,'reviewed_hits':len(rr),
       'grade2_hits':sum(r['relevance_grade']=='2' for r in rr),
       'citation_yes_hits':sum(r['citation_ok'].lower()=='yes' for r in rr),
       'citation_no_hits':sum(r['citation_ok'].lower()=='no' for r in rr),
       'rights_yes_hits':sum(r['rights_evidence_ok'].lower()=='yes' for r in rr),
       'rights_no_hits':sum(r['rights_evidence_ok'].lower()=='no' for r in rr),
       'rights_unknown_hits':sum(r['rights_evidence_ok'].lower()=='unknown' for r in rr)}
    audit.append(x)
with SOURCE_AUDIT.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0]));w.writeheader();w.writerows(audit)
for x in audit: print(x)
print('TOTAL citation_yes:',sum(x['citation_yes_hits'] for x in audit),
      'rights_yes:',sum(x['rights_yes_hits'] for x in audit))
assert sum(x['citation_yes_hits'] for x in audit)==r4['citation_ok_hits']
assert sum(x['rights_yes_hits'] for x in audit)==r4['rights_evidence_counts']['yes']
print('Saved:', SOURCE_AUDIT)


{'source_id': 'CDC-RISK-002', 'indexed_chunks': 3, 'reviewed_hits': 0, 'grade2_hits': 0, 'citation_yes_hits': 0, 'citation_no_hits': 0, 'rights_yes_hits': 0, 'rights_no_hits': 0, 'rights_unknown_hits': 0}
{'source_id': 'CDC-T2D-001', 'indexed_chunks': 4, 'reviewed_hits': 1, 'grade2_hits': 1, 'citation_yes_hits': 1, 'citation_no_hits': 0, 'rights_yes_hits': 1, 'rights_no_hits': 0, 'rights_unknown_hits': 0}
{'source_id': 'LC-HBA1C-006', 'indexed_chunks': 11, 'reviewed_hits': 6, 'grade2_hits': 3, 'citation_yes_hits': 4, 'citation_no_hits': 2, 'rights_yes_hits': 0, 'rights_no_hits': 6, 'rights_unknown_hits': 0}
{'source_id': 'LC-STAGES-007', 'indexed_chunks': 13, 'reviewed_hits': 21, 'grade2_hits': 7, 'citation_yes_hits': 8, 'citation_no_hits': 13, 'rights_yes_hits': 0, 'rights_no_hits': 21, 'rights_unknown_hits': 0}
{'source_id': 'NIDDK-DIAG-003', 'indexed_chunks': 19, 'reviewed_hits': 4, 'grade2_hits': 2, 'citation_yes_hits': 4, 'citation_no_hits': 0, 'rights_yes_hits': 4, 'rights_no_hit

## 2. Đánh giá những câu có strict hit = 0
Xem lại Q01/Q05; không sửa bảng chấm của Bước 4.

In [3]:
for qid in ('Q01','Q05'):
    per=next(q for q in r4['per_query'] if q['query_id']==qid)
    print('\n',qid,'grades:',per['graded_hits'],'strict_hit:',per['strict_hit_at_3'],'top1:',per['top1_source'])
    for r in sorted((r for r in rows if r['query_id']==qid),key=lambda x:int(x['rank'])):
        print('  rank',r['rank'],'source',r['source_id'],'grade',r['relevance_grade'],
              'citation',r['citation_ok'],'rights',r['rights_evidence_ok'],
              'section',r.get('section',''))



 Q01 grades: [0, 1, 1] strict_hit: 0 top1: LC-STAGES-007
  rank 1 source LC-STAGES-007 grade 0 citation no rights no section Cách chẩn đoán các giai đoạn tiểu đường
  rank 2 source LC-STAGES-007 grade 1 citation no rights no section Cách phòng ngừa bệnh tiểu đường tiến triển
  rank 3 source LC-STAGES-007 grade 1 citation no rights no section Cách phòng ngừa bệnh tiểu đường tiến triển

 Q05 grades: [1, 1, 1] strict_hit: 0 top1: NIDDK-DIAG-003
  rank 1 source NIDDK-DIAG-003 grade 1 citation yes rights yes section Overview
  rank 2 source NIDDK-T2D-004 grade 1 citation yes rights yes section How do health care professionals diagnose type 2 diabetes?
  rank 3 source NIDDK-DIAG-003 grade 1 citation yes rights yes section Random plasma glucose test


## 3. Tạo biểu mẫu kiểm tra quyền cụ thể cho từng nguồn (không tự duyệt)

Chỉ xác minh quyền sau khi bạn đã đối chiếu chính xác **văn bản đang index** với giấy phép hoặc thư cho phép. Một URL nguồn công khai không tự động cấp quyền sao chép và xử lý toàn văn bằng LLM.

In [4]:
source_ids=sorted(r3['sources_indexed'])
if RIGHTS_TEMPLATE.exists():
    print('Template đã tồn tại; KHÔNG ghi đè:',RIGHTS_TEMPLATE)
else:
    fields=['source_id','rights_review_status','content_origin','evidence_url_or_permission',
            'reviewer','reviewed_at_utc','medical_content_review_status','notes']
    with RIGHTS_TEMPLATE.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
        for s in source_ids:
            w.writerow({'source_id':s,'rights_review_status':'pending',
                       'medical_content_review_status':'pending',
                       'notes':'Không coi nhãn licensed_source_text là bằng chứng quyền'})
    print('Template created:',RIGHTS_TEMPLATE)
with RIGHTS_TEMPLATE.open(encoding='utf-8-sig',newline='') as f:
    rights_rows=list(csv.DictReader(f))
assert {r['source_id'] for r in rights_rows}==set(source_ids), 'Chưa khớp đủ 7 source_id'
for r in rights_rows:
    print(r['source_id'],'rights:',r['rights_review_status'],
          'medical:',r['medical_content_review_status'],
          'evidence:',bool(r.get('evidence_url_or_permission','').strip()))


Template đã tồn tại; KHÔNG ghi đè: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\registry\step5a_source_rights_review.csv
CDC-T2D-001 rights: pending medical: pending evidence: False
CDC-RISK-002 rights: pending medical: pending evidence: False
NIDDK-DIAG-003 rights: pending medical: pending evidence: False
NIDDK-T2D-004 rights: pending medical: pending evidence: False
WHO-DIAB-005 rights: pending medical: pending evidence: False
LC-HBA1C-006 rights: pending medical: pending evidence: False
LC-STAGES-007 rights: pending medical: pending evidence: False


## 4. Tùy chọn: mở rộng Top 3 → Top 10 chỉ để điều tra Q01/Q05

Cell này nạp lại **embedding model**, chỉ **query** collection cũ, không ghi ChromaDB. Có thể mất thời gian khi chạy CPU lần đầu. Đây là phép thử thăm dò trên development benchmark; không được dùng như external validation.

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
path_report=Path(r3['vector_store_path'])
vector_path=path_report if path_report.exists() else ROOT/'data'/'vector_store'/'chroma_step3'
assert vector_path.exists(),f'Không thấy vector store: {vector_path}'
client=chromadb.PersistentClient(path=str(vector_path))
collection=client.get_collection(r3['collection'])
assert collection.count()==66
assert (collection.metadata or {}).get('model_id')==r3['embedding_model']
questions={qid:next(x['query'] for x in rows if x['query_id']==qid) for qid in ('Q01','Q05')}
embedding_model=SentenceTransformer(r3['embedding_model'],device='cpu')
assert embedding_model.get_sentence_embedding_dimension()==r3['embedding_dimension']==384
q_emb=embedding_model.encode(['query: '+questions[qid] for qid in ('Q01','Q05')],normalize_embeddings=True,convert_to_numpy=True)
assert np.isfinite(q_emb).all()
res=collection.query(query_embeddings=q_emb.tolist(),n_results=10,
                     include=['metadatas','documents','distances'])
if TOP10_CSV.exists():
    print('Top10 review đã tồn tại; KHÔNG ghi đè:',TOP10_CSV)
else:
    output=[]
    for qi,qid in enumerate(('Q01','Q05')):
        for rank,(item_id,metadata,doc,dist) in enumerate(zip(res['ids'][qi],res['metadatas'][qi],res['documents'][qi],res['distances'][qi]),1):
            output.append({'query_id':qid,'query':questions[qid],'rank':rank,'chunk_id':item_id,
                'source_id':metadata.get('source_id',''),'section':metadata.get('section',''),
                'url':metadata.get('canonical_url',''),
                'cosine_distance':round(float(dist),6),
                'preview_120chars':' '.join((doc or '').split())[:120],
                'relevance_grade':'','review_notes':''})
    with TOP10_CSV.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(output[0]));w.writeheader();w.writerows(output)
    print('Saved top10 candidates:',TOP10_CSV)
for qi,qid in enumerate(('Q01','Q05')):
    print('\n',qid,questions[qid])
    for rank,(m,d) in enumerate(zip(res['metadatas'][qi],res['distances'][qi]),1):
        print(rank,m.get('source_id'),m.get('section'),round(float(d),4))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved top10 candidates: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\outputs\rag_gate_step5a\step5a_q01_q05_top10_review.csv

 Q01 Những yếu tố nào làm tăng nguy cơ mắc tiểu đường type 2?
1 LC-STAGES-007 Cách chẩn đoán các giai đoạn tiểu đường 0.0979
2 LC-STAGES-007 Cách phòng ngừa bệnh tiểu đường tiến triển 0.0981
3 LC-STAGES-007 Cách phòng ngừa bệnh tiểu đường tiến triển 0.1011
4 LC-HBA1C-006 HbA1c bao nhiêu là tiền tiểu đường? 0.1052
5 LC-STAGES-007 Giai đoạn 3: Bệnh tiểu đường tuýp 2 0.1139
6 LC-HBA1C-006 Kết luận 0.1164
7 LC-STAGES-007 Cách phòng ngừa bệnh tiểu đường tiến triển 0.1178
8 LC-STAGES-007 Giai đoạn 4: Tiểu đường tuýp 2 có biến chứng 0.1216
9 LC-STAGES-007 Cách chẩn đoán các giai đoạn tiểu đường 0.1219
10 LC-STAGES-007 Tiểu đường có mấy giai đoạn? 0.1272

 Q05 Which blood tests are used to diagnose diabetes?
1 NIDDK-DIAG-003 Overview 0.1026
2 NIDDK-T2D-004 How do health

C:\Users\ThinkPad\AppData\Local\Temp\ipykernel_21352\2956501344.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert embedding_model.get_sentence_embedding_dimension()==r3['embedding_dimension']==384


## 5. Báo cáo preflight và khóa gọi LLM

Không tự động chuyển sang `ready_for_llm=true` kể cả nếu bạn đánh dấu verified trong CSV. Cần thêm đối chiếu quyền và kiểm định y khoa bên ngoài notebook. Ô này KHÔNG gọi nhà cung cấp LLM.

In [6]:
with RIGHTS_TEMPLATE.open(encoding='utf-8-sig',newline='') as f:
    reviewed=list(csv.DictReader(f))
confirmed=[r['source_id'] for r in reviewed if r.get('rights_review_status','').strip().lower()=='verified'
           and r.get('medical_content_review_status','').strip().lower()=='verified'
           and r.get('evidence_url_or_permission','').strip()
           and r.get('reviewer','').strip() and r.get('reviewed_at_utc','').strip()]
gate={'experiment':'EXP-RAG-GATE-STEP5A-001',
      'stage':'phase4_step5a_evidence_gate',
      'created_at_utc':datetime.now(timezone.utc).isoformat(),
      'collection':r3['collection'], 'source_snapshot_sha256':r3['source_snapshot_sha256'],
      'indexed_records_unchanged':66,'reviewed_hits_step4':42,'questions_step4':14,
      'precision_at_3_dev':r4['metrics_in_scope']['precision_at_3'],
      'strict_hit_at_3_dev':r4['metrics_in_scope']['strict_hit_at_3'],
      'mrr_at_3_dev':r4['metrics_in_scope']['mrr_at_3'],
      'citation_ok_hits':r4['citation_ok_hits'],
      'rights_evidence_counts':r4['rights_evidence_counts'],
      'source_rights_and_medical_review_claimed_complete':confirmed,
      'source_reviews_independent':False,
      'ready_for_llm':False,
      'blockers':['Rights/permissions require independent source-level verification',
                  'Citation correctness only 25/42 reviewed hits',
                  'Q01 and Q05 have no grade-2 hits at top3',
                  'No tested abstention, medical safety, or prompt-injection defenses'],
      'top10_review_file_exists':TOP10_CSV.exists(),
      'llm_called':False,'embedding_index_modified':False,'model_retrained':False}
GATE_JSON.write_text(json.dumps(gate,ensure_ascii=False,indent=2),encoding='utf-8')
print('===== GIAI ĐOẠN 4 — BƯỚC 5A: EVIDENCE GATE =====')
print('Indexed chunks:',gate['indexed_records_unchanged'],'| Sources:',len(source_ids))
print('Baseline P@3:',gate['precision_at_3_dev'],'| Strict Hit@3:',gate['strict_hit_at_3_dev'])
print('Citation OK:',gate['citation_ok_hits'],'/ 42 | Rights yes:',gate['rights_evidence_counts'].get('yes'),'/ 42')
print('Verified claims (require independent checking):',len(confirmed),'/',len(source_ids))
print('Ready for LLM:',gate['ready_for_llm'])
print('Index modified: NO | LLM: NO | ML train: NO')
print('Saved:',GATE_JSON)
print('===== KẾT THÚC — GỬI REPORT CHO TÔI =====')


===== GIAI ĐOẠN 4 — BƯỚC 5A: EVIDENCE GATE =====
Indexed chunks: 66 | Sources: 7
Baseline P@3: 0.8611 | Strict Hit@3: 0.8333
Citation OK: 25 / 42 | Rights yes: 13 / 42
Verified claims (require independent checking): 0 / 7
Ready for LLM: False
Index modified: NO | LLM: NO | ML train: NO
Saved: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\Giai-doan-4-Bo-sung-Nha-thuoc-Long-Chau\Giai-doan-4-Buoc-2-Jupyter-Document-Preparation\data\outputs\rag_gate_step5a\step5a_gate_report.json
===== KẾT THÚC — GỬI REPORT CHO TÔI =====
